# Práctica final — Agente investigador sobre informes 10-K

Este notebook **integra** lo construido en las sesiones 1 y 2 (`S1_Herramientas_y_Bucle_Alumno.ipynb`
y `S2_Robustez_y_Evaluacion_Alumno.ipynb`) en el sistema final que pide el enunciado
(`Practica_LLM_Agente_10K.docx`): las 4 herramientas con el retrieval mejorado, el agente
con sus guardrails, los 3 evaluadores, `responder()`/`evaluar()` con el contrato exacto del
enunciado, y la tabla baseline-contra-final con coste y latencia.

Es autocontenido: no depende de `miax_s1.py` ni `miax_s2.py` (esos eran andamiaje de clase).
Solo necesita `corpus/` y `golden_set.jsonl`, en esta misma carpeta.

**Qué mejora trae el sistema "final" frente al "baseline" (día 10), y por qué:**

| Mejora | De dónde sale | Medido |
| --- | --- | --- |
| Filtro de metadatos en `search_filings` | Arreglo 1 (sesión 2) | recall@5: 30,8 % → 46,2 % |
| Reescritura de la consulta a inglés | Arreglo 3 (sesión 2) | recall@5: 46,2 % → **92,3 %** |
| `ToolCallLimitMiddleware` + `ModelCallLimitMiddleware` | Sesión 2 | corta el bucle infinito del día 10 |
| `verificar_cifras_contra_xbrl` | Sesión 2 | corrige cifras que no cuadran con XBRL |
| `get_xbrl_fact` resuelve el alias del concepto de ingresos | Mejora de herramienta | por medir en la tabla final |

El híbrido BM25 (Arreglo 2) **no** se incluye: medido sobre el golden set oficial no
aportó nada por encima del filtro + reescritura (mismo recall@5, con una dependencia
más) — es justo la conclusión de la celda "El arreglo que no arregló nada" de la sesión 2.


In [44]:
# Instalación. Misma celda que en las sesiones 1 y 2, versiones fijadas.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-openrouter==0.2.8 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0
print("Instalación terminada.")


Note: you may need to restart the kernel to use updated packages.
Instalación terminada.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
# Claves. Se leen del entorno; si no están, se piden por teclado sin
# que queden en la salida de la celda.
import os
import getpass


def pedir_clave(nombre: str, donde: str) -> bool:
    if os.environ.get(nombre):
        print(f"{nombre}: ya estaba en el entorno.")
        return True
    try:
        valor = getpass.getpass(f"{nombre} (se saca en {donde}): ").strip()
    except Exception:
        valor = ""
    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: guardada en el entorno de esta sesión.")
        return True
    print(f"{nombre}: sin clave. Las celdas que llaman al modelo no van a funcionar.")
    return False


HAY_CLAVE = pedir_clave("OPENROUTER_API_KEY", "openrouter.ai/keys")


OPENROUTER_API_KEY: ya estaba en el entorno.


In [46]:
# Corpus e índice. Descomprime los ZIP (si aún no está hecho) y verifica
# que el índice y los metadatos están alineados.
import hashlib
import pathlib
import zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
DESTINO = pathlib.Path("corpus")


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


if (DESTINO / "chunks.jsonl").is_file():
    print("Corpus ya montado en", DESTINO.resolve())
else:
    for nombre, esperado in PAQUETES:
        origen = pathlib.Path(nombre)
        assert origen.is_file(), f"Falta {nombre} junto al notebook."
        obtenido = _sha256(origen)
        assert obtenido == esperado, f"{nombre} no coincide con lo esperado."
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)
    print("Corpus descomprimido en", DESTINO.resolve())

assert (DESTINO / "chunks.jsonl").is_file()
assert (DESTINO / "indice" / "corpus.faiss").is_file()
print("Corpus e índice verificados.")


Corpus ya montado en C:\Users\Joseph\Documents\MIAX\Modulo V\Taller NLP\corpus
Corpus e índice verificados.


In [47]:
# El modelo. Cambiad esta cadena para probar con uno gratuito mientras
# depuráis (p. ej. "openrouter:nvidia/nemotron-3-super-120b-a12b:free") y
# volved a uno de pago para la tabla final que vaya en el informe.
from langchain.chat_models import init_chat_model

MODELO = "openrouter:google/gemini-3.8-flash"

modelo = None
if HAY_CLAVE:
    try:
        modelo = init_chat_model(MODELO, temperature=0)
        print(modelo.invoke("Responde solo con la palabra: listo").text)
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")
else:
    print("Sin clave: el resto del notebook define el código pero no lo ejecuta.")


listo


## Datos base y motor de retrieval

Carga `secciones`, `xbrl`, el índice FAISS y sus metadatos. `_con_filtros` es el Arreglo 1
de la sesión 2 (filtro de metadatos, medido); `_reescribir` es el Arreglo 3 (traducción de
la consulta al inglés con el propio LLM, el que de verdad sube el recall).


In [48]:
import json
import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open("corpus/secciones.jsonl", encoding="utf-8"))
xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")
chunks = [json.loads(l) for l in open("corpus/chunks.jsonl", encoding="utf-8")]
POR_ID = {c["chunk_id"]: c for c in chunks}

import faiss
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "BAAI/bge-small-en-v1.5"
PREFIJO_CONSULTA_BGE = "Represent this sentence for searching relevant passages: "

_indice = faiss.read_index("corpus/indice/corpus.faiss")
_meta = pd.read_parquet("corpus/indice/chunks_meta.parquet")
_codificador = SentenceTransformer(MODELO_EMBEDDINGS)
assert _indice.ntotal == len(_meta), "índice y metadatos desalineados"


def _codificar(textos: list[str]):
    return _codificador.encode(
        [PREFIJO_CONSULTA_BGE + t for t in textos],
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")


def _fila_a_fragmento(fila, puntuacion: float) -> dict:
    return {
        "chunk_id": fila["chunk_id"], "ticker": fila["ticker"],
        "fiscal_year": int(fila["fiscal_year"]), "item": fila["item"],
        "texto": fila["texto"], "n_tokens": int(fila["n_tokens"]),
        "contiene_tabla": bool(fila["contiene_tabla"]),
        "puntuacion": round(float(puntuacion), 4),
    }


def _formatear_fragmentos(fragmentos: list[dict]) -> str:
    if not fragmentos:
        return ("Sin resultados para esa consulta con esos filtros. "
                "Prueba a quitar algún filtro o a reformular la búsqueda.")
    return "\n\n---\n\n".join(
        f"[{f['chunk_id']}] {f['ticker']} FY{f['fiscal_year']} "
        f"Item {f['item']} (similitud {f['puntuacion']:.3f})\n{f['texto']}"
        for f in fragmentos
    )


def _con_filtros(consulta: str, ticker=None, fiscal_year=None, item=None, k: int = 5) -> list[dict]:
    '''Búsqueda densa filtrada por metadatos. Arreglo 1 de la sesión 2.'''
    puntuaciones, posiciones = _indice.search(_codificar([consulta]), _indice.ntotal)
    salida = []
    for puntuacion, posicion in zip(puntuaciones[0], posiciones[0]):
        fila = _meta.iloc[int(posicion)]
        if ticker is not None and fila["ticker"] != ticker:
            continue
        if fiscal_year is not None and int(fila["fiscal_year"]) != int(fiscal_year):
            continue
        if item is not None and fila["item"] != item:
            continue
        salida.append(_fila_a_fragmento(fila, puntuacion))
        if len(salida) >= k:
            break
    return salida


INSTRUCCION_REESCRITURA = '''Reescribe esta pregunta como una consulta de búsqueda para un
índice de informes 10-K en INGLÉS. Usa el vocabulario del propio informe.
Devuelve SOLO la consulta, sin comillas ni explicación.'''

_reescritor = None
if HAY_CLAVE:
    try:
        _reescritor = init_chat_model(MODELO, temperature=0)
    except Exception as e:
        print(f"Sin reescritor en vivo ({type(e).__name__}).")


import time

REINTENTOS_LLAMADA = 4      # reintentos de UNA llamada al modelo (no de la pregunta entera)
ESPERA_INICIAL_S = 5.0      # espera creciente entre reintentos: 5, 10, 20, 40 s
ESPERA_MAX_S = 60.0

_USO_REESCRITURA = {"entrada": 0, "salida": 0, "fallos": 0}   # tokens gastados y fallos al reescribir


def _reescribir(pregunta: str) -> str:
    '''La pregunta, convertida a consulta en inglés. Arreglo 3 de la sesión 2:
    es el que de verdad sube el recall. Si la llamada falla se reintenta ELLA SOLA
    (espera creciente); si se agotan los reintentos se busca con la pregunta tal
    cual, y se anota en _USO_REESCRITURA["fallos"] para que no pase inadvertido.
    Los tokens que gasta se suman a _USO_REESCRITURA para que responder() los
    incluya en el coste.'''
    if _reescritor is None:
        return pregunta
    for intento in range(REINTENTOS_LLAMADA + 1):
        try:
            respuesta = _reescritor.invoke(
                [{"role": "system", "content": INSTRUCCION_REESCRITURA},
                 {"role": "user", "content": pregunta}]
            )
            uso = getattr(respuesta, "usage_metadata", None) or {}
            _USO_REESCRITURA["entrada"] += uso.get("input_tokens", 0) or 0
            _USO_REESCRITURA["salida"] += uso.get("output_tokens", 0) or 0
            return respuesta.text.strip()
        except Exception:
            if intento < REINTENTOS_LLAMADA:
                time.sleep(min(ESPERA_INICIAL_S * 2 ** intento, ESPERA_MAX_S))
    _USO_REESCRITURA["fallos"] += 1
    return pregunta


print(f"{len(secciones)} secciones · {len(xbrl)} hechos XBRL · "
      f"{_indice.ntotal} vectores en el índice")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3471.60it/s]


48 secciones · 135 hechos XBRL · 1749 vectores en el índice


## Las cuatro herramientas (contrato del §7 del enunciado)

Nombres y parámetros son los del enunciado, sin tocar. `search_filings` y `get_xbrl_fact` son
las que cambian de verdad respecto al día 10. `search_filings` por dentro reescribe la consulta a inglés y filtra
por metadatos antes de devolver nada — el modelo no ve esa diferencia, solo ve que
funciona mejor.

También se define `search_filings_baseline`, la versión **sin mejorar** (día 10, sin
filtro ni reescritura), que se usa más abajo solo para montar el agente *baseline* de
comparación — no forma parte del sistema final.

`get_xbrl_fact` también mejora: los ingresos se reportan con dos conceptos distintos según
la compañía (`Revenues` o `RevenueFromContractWithCustomerExcludingAssessedTax`), y ahora
la herramienta devuelve el que sí existe en vez de obligar al modelo a gastar otra llamada
equivocándose. No inventa alias en los huecos reales (Amazon sigue sin `GrossProfit`).
El baseline conserva la versión original (`get_xbrl_fact_baseline`), y las dos versiones
de las herramientas *baseline* llevan el **mismo nombre** que las finales (`@tool("...")`):
así `uso_la_tool_correcta` ve los mismos nombres en los dos sistemas y la comparación es limpia.


In [49]:
from langchain.tools import tool


@tool
def list_available() -> str:
    '''Lista qué compañías, ejercicios y secciones existen en el corpus.

    Úsala SIEMPRE antes de responder que un dato no existe, y antes de
    llamar a cualquier otra herramienta si no estás seguro de que la
    compañía o el ejercicio que te piden están en el corpus.
    '''
    lineas = ["Compañías en el corpus:"]
    for ticker, grupo in secciones.groupby("ticker"):
        empresa = grupo.iloc[0]["empresa"]
        ejercicios = sorted(grupo["fiscal_year"].unique().tolist())
        items = sorted(grupo["item"].unique().tolist())
        lineas.append(f"  {ticker} ({empresa}): ejercicios {ejercicios}, items {items}")
    return "\n".join(lineas)


INGRESOS = ("Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax")


@tool("get_xbrl_fact")
def get_xbrl_fact_baseline(ticker: str, fiscal_year: int, concept: str) -> str:
    '''Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL. Es la fuente autorizada para cualquier
    cifra. Úsala SIEMPRE en lugar de leer un número del texto.'''
    filas = xbrl[(xbrl.ticker == ticker) & (xbrl.fiscal_year == int(fiscal_year))
                 & (xbrl.concept == concept)]
    if filas.empty:
        disponibles = sorted(xbrl[(xbrl.ticker == ticker)
                                   & (xbrl.fiscal_year == int(fiscal_year))].concept.unique())
        if not disponibles:
            return f"No hay datos de {ticker} para FY{fiscal_year} en el corpus."
        return (f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {', '.join(disponibles)}")
    f = filas.iloc[0]
    return (f"{ticker} FY{fiscal_year} · {concept} = {f.value:,.0f} {f.unit} "
            f"(cierre de ejercicio {f.period_end}, según el {f.form})")


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    '''Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL.

    Es la fuente autorizada para cualquier cifra. Úsala SIEMPRE en lugar
    de leer un número del texto del informe.

    Los ingresos se reportan con DOS conceptos según la compañía:
    'Revenues' o 'RevenueFromContractWithCustomerExcludingAssessedTax'. Si
    pides uno y la compañía usa el otro, la herramienta te devuelve el que
    sí reportó e indica cuál es.

    Args:
        ticker: Símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: Ejercicio fiscal reportado, p. ej. 2024.
        concept: Concepto en taxonomía US-GAAP, p. ej. 'Revenues',
            'NetIncomeLoss', 'Assets', 'OperatingIncomeLoss'.

    Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito
    si la compañía no reportó ese concepto en ese ejercicio.
    '''
    ticker = ticker.strip().upper()
    base = xbrl[(xbrl.ticker == ticker) & (xbrl.fiscal_year == int(fiscal_year))]
    filas = base[base.concept == concept]
    aviso = ""
    if filas.empty and concept in INGRESOS:
        for alternativo in INGRESOS:
            if alternativo != concept and not base[base.concept == alternativo].empty:
                filas = base[base.concept == alternativo]
                aviso = (f" [aviso: {ticker} no usa '{concept}' en FY{fiscal_year}; "
                         f"reporta los ingresos como '{alternativo}']")
                break
    if filas.empty:
        if base.empty:
            return f"No hay datos de {ticker} para FY{fiscal_year} en el corpus."
        return (f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {', '.join(sorted(base.concept.unique()))}")
    f = filas.iloc[0]
    return (f"{ticker} FY{fiscal_year} · {f.concept} = {f.value:,.0f} {f.unit} "
            f"(cierre de ejercicio {f.period_end}, según el {f.form}){aviso}")


@tool
def search_filings(query: str, ticker: str | None = None,
                   fiscal_year: int | None = None,
                   item: str | None = None, k: int = 5) -> str:
    '''Busca fragmentos de texto relevantes en los informes 10-K del corpus.

    Úsala para preguntas cualitativas: riesgos, estrategia, litigios,
    comentarios de la dirección. NO la uses para obtener cifras: para eso
    está get_xbrl_fact.

    Args:
        query: Qué buscar, en lenguaje natural (puede venir en español: la
            propia herramienta la traduce antes de buscar).
        ticker: Filtra por compañía si la pregunta la menciona.
        fiscal_year: Filtra por ejercicio si la pregunta lo menciona.
        item: Filtra por sección: '1A' riesgos, '7' MD&A,
            '7A' riesgo de mercado, '8' estados financieros.
        k: Número de fragmentos a devolver.

    Devuelve k fragmentos, cada uno con su chunk_id para poder citarlo.
    '''
    consulta_en = _reescribir(query)
    fragmentos = _con_filtros(consulta_en, ticker, fiscal_year, item, k)
    return _formatear_fragmentos(fragmentos)


@tool("search_filings")
def search_filings_baseline(query: str, ticker: str | None = None,
                            fiscal_year: int | None = None,
                            item: str | None = None, k: int = 5) -> str:
    '''Versión SIN mejorar de search_filings (día 10): sin filtro de
    metadatos ni reescritura de consulta. Solo para el agente baseline.'''
    puntuaciones, posiciones = _indice.search(_codificar([query]), k)
    fragmentos = [_fila_a_fragmento(_meta.iloc[int(i)], s)
                  for s, i in zip(puntuaciones[0], posiciones[0])]
    return _formatear_fragmentos(fragmentos)


@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    '''Devuelve el TEXTO COMPLETO de una sección de un 10-K.

    Es una herramienta CARA: puede devolver decenas de miles de tokens.
    Úsala solo cuando search_filings devuelva fragmentos insuficientes y
    necesites el contexto entero de una sección concreta.

    Args:
        ticker: Símbolo bursátil, p. ej. 'META'.
        fiscal_year: Ejercicio fiscal, p. ej. 2025.
        item: '1A' riesgos, '7' MD&A, '7A' riesgo de mercado,
            '8' estados financieros.
    '''
    filas = secciones[(secciones.ticker == ticker)
                      & (secciones.fiscal_year == int(fiscal_year))
                      & (secciones.item == item)]
    if filas.empty:
        return f"No hay Item {item} de {ticker} FY{fiscal_year} en el corpus."
    return filas.iloc[0].texto


HERRAMIENTAS = [list_available, get_xbrl_fact, search_filings, read_section]
HERRAMIENTAS_BASELINE = [list_available, get_xbrl_fact_baseline, search_filings_baseline, read_section]
print(f"{len(HERRAMIENTAS)} herramientas finales · {len(HERRAMIENTAS_BASELINE)} de baseline")


4 herramientas finales · 4 de baseline


## El esquema de respuesta y los guardrails

`RespuestaFinanciera` es el contrato del §7 del enunciado. Los guardrails son los tres de
la sesión 2: dos límites (`ToolCallLimitMiddleware`, `ModelCallLimitMiddleware`) que cortan
el bucle infinito del día 10, y `verificar_cifras_contra_xbrl`, el middleware propio que
contrasta cada cifra que el agente dice sacar del XBRL (`fuente` `xbrl` o `ambas`)
contra `xbrl_facts.parquet` y le devuelve el desajuste al
modelo si no cuadra.


In [50]:
from typing import Literal
from pydantic import BaseModel, Field


class RespuestaFinanciera(BaseModel):
    '''Respuesta trazable a una pregunta sobre informes 10-K.'''

    respuesta: str = Field(description="Respuesta en prosa, breve y directa")
    cifra: float | None = Field(
        default=None, description="Valor numérico, si la pregunta pide uno")
    unidad: str | None = Field(default=None, description="USD, shares, porcentaje…")
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"] = Field(
        description="De dónde sale el dato. 'ninguna' si no está en el corpus")
    cita: str | None = Field(
        default=None, description="Texto literal del informe que respalda la respuesta")
    chunk_id: str | None = Field(
        default=None, description="Identificador del fragmento citado, para verificar")


In [51]:
# El middleware de verificación de cifras contra XBRL.
from langchain.agents.middleware import AgentState, after_model
from langgraph.runtime import Runtime

TOLERANCIA = 0.01          # 1 %: redondear no es inventarse un número
MARCA = "VERIFICACIÓN AUTOMÁTICA"


def _cuadra(afirmada: float, real: float, tolerancia: float = TOLERANCIA) -> bool:
    '''¿La cifra afirmada coincide con la real, con tolerancia relativa?'''
    if real == 0:
        return afirmada == 0
    return abs(afirmada - real) / abs(real) <= tolerancia


@after_model(can_jump_to=["model"])
def verificar_cifras_contra_xbrl(state: AgentState, runtime: Runtime) -> dict | None:
    '''Contrasta la cifra de la respuesta con el XBRL. Si no cuadra, se lo
    devuelve al modelo para que se corrija. Solo verifica las cifras que el
    agente dice sacar del XBRL (fuente 'xbrl' o 'ambas'): una cifra de la prosa
    del informe (p. ej. el precio de una adquisición) no es un hecho XBRL, y
    contrastarla sería un falso positivo.'''
    respuesta = state.get("structured_response")
    if respuesta is None or respuesta.cifra is None:
        return None
    if respuesta.fuente not in ("xbrl", "ambas"):
        return None

    ticker, ejercicio = respuesta.ticker, respuesta.ejercicio
    if ticker is None or ejercicio is None:
        return None

    if any(MARCA in str(getattr(m, "content", "")) for m in state["messages"]):
        return None  # una corrección por invocación: sin esto, ping-pong infinito

    hechos = xbrl[(xbrl.ticker == ticker) & (xbrl.fiscal_year == int(ejercicio))]
    if any(_cuadra(respuesta.cifra, real) for real in hechos["value"]):
        return None

    reportados = ", ".join(
        f"{row.concept}={row.value:,.2f} {row.unit}" for row in hechos.itertuples()
    ) or "ningún dato XBRL para esa compañía y ejercicio"

    return {
        "messages": [{
            "role": "user",
            "content": (
                f"{MARCA}: afirmaste la cifra {respuesta.cifra:,.0f} "
                f"{respuesta.unidad or ''} para {ticker} FY{ejercicio}, pero no "
                f"coincide (tolerancia {TOLERANCIA:.0%}) con ningún hecho reportado "
                f"en XBRL. Lo que SÍ está reportado: {reportados}. Corrige la cifra "
                f"usando get_xbrl_fact, o si el concepto no está entre los "
                f"reportados, di explícitamente que no está en el corpus. No la "
                f"estimes."
            ),
        }],
        "jump_to": "model",
    }


print("Middleware de verificación definido.")


Middleware de verificación definido.


## Los dos agentes: baseline (día 10) y final

**Baseline**: las 4 herramientas del día 10, `search_filings` sin mejorar, sin guardrails.
Es el sistema de partida contra el que se compara.

**Final**: `search_filings` con filtro + reescritura, `get_xbrl_fact` con alias de ingresos,
y los tres guardrails de la sesión 2.

**Común a los dos (infraestructura, no mejora):** el mismo modelo con `temperature=0` y un
`ModelRetryMiddleware` que, si falla UNA llamada al modelo (límite de peticiones por minuto,
error del proveedor), reintenta solo esa llamada con espera creciente (5, 10, 20, 40 s). No
repite lo ya pagado ni cuenta contra `ModelCallLimitMiddleware`. Si se agotan los reintentos,
la excepción queda registrada en la columna `error`. Sin esto, un límite de velocidad tira
media tanda y la comparación deja de medir el sistema.


In [52]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ModelRetryMiddleware, ToolCallLimitMiddleware)
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM = '''Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas:
- Para cualquier CIFRA, usa get_xbrl_fact. Nunca leas un número de la prosa.
- Para riesgos, estrategia o comentarios de la dirección, usa search_filings.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Cita el chunk_id del fragmento en el que te apoyes.
- Si el dato no está en el corpus, dilo. No lo estimes.
'''

agente_baseline = None
agente_final = None
if HAY_CLAVE:
    # El mismo modelo para los dos agentes, con temperature=0 (todo lo que se evalúa la lleva).
    modelo_agente = init_chat_model(MODELO, temperature=0)

    def _reintento_por_llamada():
        '''Infraestructura común a los dos agentes, NO una mejora del sistema: si una
        llamada al modelo falla (límite de peticiones por minuto, error del proveedor)
        se reintenta SOLO esa llamada, con espera creciente (5, 10, 20, 40 s). No
        repite lo ya pagado ni cuenta contra ModelCallLimitMiddleware. Si se agotan los
        reintentos relanza la excepción, que queda en la columna `error`.'''
        return ModelRetryMiddleware(max_retries=4, initial_delay=5.0, backoff_factor=2.0,
                                    max_delay=60.0, jitter=True, on_failure="error")

    agente_baseline = create_agent(
        model=modelo_agente, tools=HERRAMIENTAS_BASELINE, system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        middleware=[_reintento_por_llamada()],
        checkpointer=InMemorySaver(),
    )
    agente_final = create_agent(
        model=modelo_agente, tools=HERRAMIENTAS, system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        middleware=[
            _reintento_por_llamada(),
            ToolCallLimitMiddleware(run_limit=8),
            ModelCallLimitMiddleware(run_limit=10),
            verificar_cifras_contra_xbrl,
        ],
        checkpointer=InMemorySaver(),
    )
    print("Agentes baseline y final montados.")
else:
    print("Sin clave: no se pueden montar los agentes todavía.")


Agentes baseline y final montados.


## `responder()`, trazas, coste y latencia


In [53]:
import time
import uuid

PRECIOS_OPENROUTER = {
    # USD por millón de tokens (entrada, salida). Revisar la víspera de entregar.
    "google/gemini-3.5-flash-lite":  (0.30,  2.50),
    "google/gemini-3.8-flash":       (0.75,  3.75),
    "anthropic/claude-opus-5":       (5.00, 25.00),
    "anthropic/claude-fable-5.1":   (10.00, 50.00),
}

HERRAMIENTAS_REALES = {"list_available", "get_xbrl_fact", "search_filings",
                       "read_section"}


def _tokens_de(resultado: dict) -> tuple[int, int]:
    entrada = salida = 0
    for m in resultado.get("messages") or []:
        uso = getattr(m, "usage_metadata", None) or {}
        entrada += uso.get("input_tokens", 0) or 0
        salida += uso.get("output_tokens", 0) or 0
    return entrada, salida


def _coste_tokens(entrada: int, salida: int, modelo: str) -> float:
    nombre = modelo.split(":", 1)[-1]
    if nombre not in PRECIOS_OPENROUTER:
        return 0.0
    p_in, p_out = PRECIOS_OPENROUTER[nombre]
    return (entrada * p_in + salida * p_out) / 1e6


def _coste_de(resultado: dict, modelo: str) -> float:
    '''Coste de los mensajes del agente (sin las reescrituras de consulta).'''
    entrada, salida = _tokens_de(resultado)
    return _coste_tokens(entrada, salida, modelo)


def herramientas_usadas(resultado: dict) -> list[str]:
    '''Nombres de herramientas de DOMINIO en la trayectoria (excluye la tool
    sintética que create_agent usa por dentro para la salida estructurada).'''
    return [tc["name"] for m in resultado.get("messages") or []
            for tc in (getattr(m, "tool_calls", None) or [])
            if tc["name"] in HERRAMIENTAS_REALES]


def pretty_trace(resultado: dict, max_chars: int = 200) -> None:
    '''Qué herramientas se llamaron, con qué argumentos y qué devolvieron.'''
    pendientes = {}
    n = 0
    for m in resultado["messages"]:
        for tc in getattr(m, "tool_calls", None) or []:
            n += 1
            pendientes[tc["id"]] = n
            print(f"  {n}. {tc['name']}({tc['args']})")
        id_llamada = getattr(m, "tool_call_id", None)
        if id_llamada in pendientes:
            texto = str(m.content).replace("\n", " ")
            print(f"       -> {texto[:max_chars]}{'...' if len(texto) > max_chars else ''}")
    e = resultado.get("structured_response")
    if e is not None:
        print(f"\n  respuesta: {e.respuesta}")
        print(f"  fuente: {e.fuente} · cifra: {e.cifra} {e.unidad or ''} · cita: {e.chunk_id}")


def responder(pregunta: str, agente=None, thread_id: str | None = None) -> dict:
    '''Le hace una pregunta suelta al agente (el FINAL por defecto) y devuelve
    el resultado completo de LangGraph, con latencia, coste y llamadas a
    herramienta añadidos. El coste incluye las llamadas de reescritura de
    consulta que hace search_filings por dentro. Cada llamada usa un thread_id
    nuevo salvo que se pase uno explícito (para conversaciones de seguimiento).'''
    agente = agente if agente is not None else agente_final
    _USO_REESCRITURA["entrada"] = _USO_REESCRITURA["salida"] = _USO_REESCRITURA["fallos"] = 0
    inicio = time.perf_counter()
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": thread_id or str(uuid.uuid4())}},
    )
    resultado["latencia_s"] = time.perf_counter() - inicio
    coste_reescritura = _coste_tokens(
        _USO_REESCRITURA["entrada"], _USO_REESCRITURA["salida"], MODELO)
    resultado["coste_reescritura_usd"] = coste_reescritura
    resultado["reescrituras_fallidas"] = _USO_REESCRITURA["fallos"]
    resultado["coste_usd"] = _coste_de(resultado, MODELO) + coste_reescritura
    resultado["llamadas_herramienta"] = len(herramientas_usadas(resultado))
    return resultado


## Los tres evaluadores

`cita_correcta`, `cifra_coincide_xbrl` y `uso_la_tool_correcta`, tal y como se escribieron
en la sesión 2 (probados con casos sintéticos sin gastar llamadas al modelo).


In [54]:
import re

_ESPACIOS = re.compile(r"\s+")


def _normalizar(texto: str) -> str:
    return _ESPACIOS.sub(" ", texto).strip().lower()


def cita_correcta(item: dict, resultado: dict) -> bool | None:
    '''El chunk_id citado existe y su texto contiene lo que dice la cita.
    Si el agente no devolvió respuesta estructurada, cuenta como fallo cuando
    la pregunta exigía cita (extractiva o comparativa).'''
    e = resultado.get("structured_response")
    requiere_cita = item["familia"] != "numerica"  # extractiva y comparativa

    if e is None or not e.chunk_id:
        return False if requiere_cita else None

    fragmento = POR_ID.get(e.chunk_id)
    if fragmento is None:
        return False  # citó un chunk_id que no existe: se lo inventó

    if not e.cita:
        return False

    cita = _normalizar(e.cita)[:120]
    return cita in _normalizar(fragmento["texto"])


def cifra_coincide_xbrl(item: dict, resultado: dict) -> bool | None:
    '''La cifra afirmada coincide con la esperada, con tolerancia. Sin
    respuesta estructurada o sin cifra, cuenta como fallo.'''
    if item.get("cifra_esperada") is None:
        return None
    e = resultado.get("structured_response")
    if e is None or e.cifra is None:
        return False
    return _cuadra(e.cifra, item["cifra_esperada"])


def uso_la_tool_correcta(item: dict, resultado: dict) -> bool:
    '''La trayectoria pasó por TODAS las herramientas esperadas.'''
    usadas = set(herramientas_usadas(resultado))
    esperadas = set(item.get("herramienta_esperada") or [])
    return esperadas.issubset(usadas)


EVALUADORES = {
    "cita": cita_correcta,
    "cifra": cifra_coincide_xbrl,
    "trayectoria": uso_la_tool_correcta,
}
print("Evaluadores definidos:", list(EVALUADORES))


Evaluadores definidos: ['cita', 'cifra', 'trayectoria']


## `evaluar(ruta_jsonl)` — el contrato del enunciado

Firma de un solo argumento, tal y como la pide el §6: `evaluar("holdout.jsonl")` tiene que
funcionar sobre un clon limpio del repositorio el día 24, sin tocar código. Por dentro usa
`responder()` con el agente **final** salvo que se le pase explícitamente `agente_baseline`
(así es como se genera la comparación de la última celda).

**Robustez.** Una pregunta que falla **no se reintenta** (`INTENTOS = 1`), para no gastar
crédito en repeticiones; sube `INTENTOS` si prefieres reintentar errores transitorios del
proveedor. Los fallos transitorios de una *llamada* al modelo sí se reintentan, pero por
dentro del agente (`ModelRetryMiddleware`, ver más arriba), sin repetir lo ya pagado. Un fallo
**cuenta como fallo** en las métricas que le aplican (no
desaparece de los promedios) y el error queda en la columna `error`; la tabla final
muestra cuántos hubo. Cuenta como error tanto una excepción como que el agente termine
sin respuesta estructurada (`SinRespuestaEstructurada`). Los evaluadores no lanzan
excepción: sin respuesta estructurada
devuelven `False`. El `recall@5` se mide aparte del agente, y el coste incluye las
llamadas de reescritura de consulta. La tabla guarda además `cifra_obtenida`, `fuente`,
`chunk_id` y `herramientas` para poder clasificar los fallos. El texto de las respuestas
no va en esa tabla: con `salida_respuestas="ruta.csv"` se guarda aparte (pregunta,
respuesta esperada y respuesta del agente).


In [55]:
def acierta(item_golden: dict, fragmentos: list[dict]) -> bool | None:
    '''¿Alguno de los fragmentos recuperados contiene el ancla_texto entera?
    None si la pregunta no lleva ancla (las numéricas puras).'''
    ancla = item_golden.get("ancla_texto")
    if not ancla:
        return None
    objetivo = _normalizar(ancla)
    for f in fragmentos:
        mismo_doc = (f.get("ticker") == item_golden.get("ticker")
                    and int(f.get("fiscal_year", -1)) == int(item_golden.get("fiscal_year", -2)))
        if mismo_doc and objetivo in _normalizar(f.get("texto", "")):
            return True
    return False


def _recuperar_para_recall(item: dict, es_baseline: bool) -> list[dict]:
    '''El retrieval que corresponde a cada sistema, para medir recall@5
    de forma comparable entre baseline y final.'''
    if es_baseline:
        puntuaciones, posiciones = _indice.search(_codificar([item["pregunta"]]), 5)
        return [_fila_a_fragmento(_meta.iloc[int(i)], s)
                for s, i in zip(puntuaciones[0], posiciones[0])]
    consulta_en = _reescribir(item["pregunta"])
    return _con_filtros(consulta_en, item["ticker"], item["fiscal_year"],
                        item["item_esperado"], k=5)


INTENTOS = 1   # sin reintentos: un intento fallido puede haber gastado ya crédito
PAUSA_REINTENTO_S = 3.0


def _con_reintentos(funcion, intentos: int = INTENTOS):
    '''Ejecuta funcion(n) y la reintenta si lanza una excepción (errores
    transitorios del proveedor). Devuelve (resultado, n_intentos, error).'''
    ultimo_error = None
    for n in range(1, intentos + 1):
        try:
            return funcion(n), n, None
        except Exception as e:
            ultimo_error = f"{type(e).__name__}: {e}"
            if n < intentos:
                time.sleep(PAUSA_REINTENTO_S * n)
    return None, intentos, ultimo_error


def evaluar(ruta_jsonl: str, agente=None, etiqueta: str = "eval",
            salida_respuestas: str | None = None) -> pd.DataFrame:
    '''Ejecuta responder() sobre cada pregunta del golden set en ruta_jsonl y
    aplica los tres evaluadores más recall@5. Esta es la función que se
    ejecuta el día 24 contra holdout.jsonl.

    - Por defecto NO se reintenta (INTENTOS = 1) para no gastar crédito; sube
      INTENTOS si prefieres reintentar los errores del proveedor.
    - Si falla, cuenta como FALLO en las métricas que le aplican (no
      desaparece de los promedios) y el error queda en la columna `error`.
    - El recall@5 se mide aparte, así que no se pierde si el agente falla.
    - La tabla guarda lo necesario para clasificar los fallos (cifra_obtenida,
      fuente, chunk_id, herramientas). El TEXTO de las respuestas no va en la
      tabla: si se pasa `salida_respuestas`, se escribe aparte en ese CSV.'''
    agente_efectivo = agente if agente is not None else agente_final
    es_baseline = agente_efectivo is agente_baseline

    preguntas = [json.loads(l) for l in open(ruta_jsonl, encoding="utf-8") if l.strip()]
    filas = []
    respuestas = []
    for i, item in enumerate(preguntas, 1):
        print(f"  [{i}/{len(preguntas)}] {item['id']}" + " " * 20, end="\r")
        fila = {"id": item["id"], "familia": item["familia"], "ticker": item["ticker"]}

        if item.get("ancla_texto"):
            fallos_antes = _USO_REESCRITURA["fallos"]
            fragmentos, _, error_recall = _con_reintentos(
                lambda n: _recuperar_para_recall(item, es_baseline))
            if error_recall is None:
                fila["recall"] = acierta(item, fragmentos)
                if _USO_REESCRITURA["fallos"] > fallos_antes:
                    fila["aviso_recall"] = "reescritura fallida: recall medido con la pregunta sin reescribir"
            else:
                fila["error_recall"] = error_recall

        r, intentos, error = _con_reintentos(
            lambda n: responder(item["pregunta"], agente=agente_efectivo,
                                thread_id=f"{etiqueta}-{item['id']}-{n}"))
        fila["intentos"] = intentos
        if error is None:
            fila["latencia_s"] = r["latencia_s"]
            fila["coste_usd"] = r["coste_usd"]
            fila["llamadas"] = r["llamadas_herramienta"]
            fila["reescrituras_fallidas"] = r.get("reescrituras_fallidas", 0)
            if r.get("structured_response") is None:
                # sin excepción, pero el agente terminó sin contestar en el formato
                fila["error"] = "SinRespuestaEstructurada: el agente terminó sin respuesta estructurada"
        else:
            fila["error"] = error
            r = {"messages": [], "structured_response": None}   # cuenta como fallo

        e = r.get("structured_response")
        fila["cifra_obtenida"] = e.cifra if e is not None else None
        fila["fuente"] = e.fuente if e is not None else None
        fila["chunk_id"] = e.chunk_id if e is not None else None
        fila["herramientas"] = ",".join(herramientas_usadas(r))
        respuestas.append({
            "id": item["id"], "familia": item["familia"], "ticker": item["ticker"],
            "pregunta": item["pregunta"],
            "respuesta_esperada": item.get("respuesta_esperada"),
            "respuesta": e.respuesta if e is not None else None,
        })

        for nombre, evaluador in EVALUADORES.items():
            try:
                fila[nombre] = evaluador(item, r)
            except Exception as ex:
                fila[nombre] = False
                fila["error_evaluador"] = f"{nombre}: {type(ex).__name__}: {ex}"
        filas.append(fila)
    print(" " * 40, end="\r")
    if salida_respuestas:
        pd.DataFrame(respuestas).to_csv(salida_respuestas, index=False)
    return pd.DataFrame(filas)


def resumir(tabla: pd.DataFrame, etiqueta: str) -> dict:
    '''Una fila de la tabla baseline-contra-final del informe. Las tasas se
    calculan sobre todas las preguntas a las que aplica cada evaluador (los
    errores cuentan como fallo); coste, latencia y llamadas, sobre las que
    tienen medida (las que lanzaron una excepción no la tienen).'''
    def tasa(columna):
        valores = tabla[columna].dropna() if columna in tabla else []
        return float(valores.astype(bool).mean()) if len(valores) else float("nan")

    return {
        "versión": etiqueta,
        "n": len(tabla),
        "errores": int(tabla["error"].notna().sum()) if "error" in tabla else 0,
        "reescrituras fallidas": (int(tabla["reescrituras_fallidas"].sum())
                                  if "reescrituras_fallidas" in tabla else 0),
        "cita": tasa("cita"), "cifra": tasa("cifra"), "trayectoria": tasa("trayectoria"),
        "recall@5": tasa("recall"),
        "coste medio (¢)": (tabla["coste_usd"].mean() * 100
                            if "coste_usd" in tabla else float("nan")),
        "latencia media (s)": (tabla["latencia_s"].mean()
                               if "latencia_s" in tabla else float("nan")),
        "llamadas/pregunta": (tabla["llamadas"].mean()
                              if "llamadas" in tabla else float("nan")),
    }


print("evaluar() y resumir() listos.")


evaluar() y resumir() listos.


## El golden set propio (20 preguntas, ≥6 comparativas)

Ya construido y validado (ver sesión de creación): anclado a texto y cifras reales del
corpus, no a `chunk_id` (que cambiaría en cuanto se re-troceara el corpus).


In [56]:
from pathlib import Path

RUTA_GOLDEN = Path("golden_set.jsonl")
golden = [json.loads(l) for l in open(RUTA_GOLDEN, encoding="utf-8") if l.strip()]
g = pd.DataFrame(golden)
print(f"{RUTA_GOLDEN.name}: {len(g)} preguntas")
print(g.familia.value_counts().to_string())

PLANTILLA_CAMPOS = {
    "id", "pregunta", "familia", "ticker", "fiscal_year", "respuesta_esperada",
    "cifra_esperada", "unidad", "concept_xbrl", "item_esperado", "ancla_texto",
    "ancla_inicio", "ancla_fin", "chunk_id_esperado", "herramienta_esperada", "autor",
}
FAMILIAS = {"extractiva", "numerica", "comparativa"}


def validar(preguntas: list[dict], exigir_20: bool = True) -> list[str]:
    '''Los problemas del fichero, uno por línea. Lista vacía = correcto.'''
    problemas = []
    tickers = set(secciones.ticker)
    ejercicios = set(secciones.fiscal_year.astype(int))
    vistos = set()

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        if faltan := PLANTILLA_CAMPOS - set(p):
            problemas.append(f"{pid}: faltan campos {sorted(faltan)}")
            continue
        if p["id"] in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(p["id"])
        if p["familia"] not in FAMILIAS:
            problemas.append(f"{pid}: familia '{p['familia']}' no válida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: {p['ticker']} no está en el corpus")
        if int(p["fiscal_year"]) not in ejercicios:
            problemas.append(f"{pid}: FY{p['fiscal_year']} no está en el corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p.get("cifra_esperada") is None:
                problemas.append(f"{pid}: numérica sin cifra_esperada")
            concepto = p.get("concept_xbrl")
            hay = xbrl[(xbrl.ticker == p["ticker"])
                       & (xbrl.fiscal_year == int(p["fiscal_year"]))
                       & (xbrl.concept == concepto)]
            if concepto and hay.empty:
                problemas.append(f"{pid}: {p['ticker']} no reporta '{concepto}' en "
                                 f"FY{p['fiscal_year']}.")
        if p["familia"] in {"extractiva", "comparativa"}:
            ancla = p.get("ancla_texto")
            if not ancla:
                problemas.append(f"{pid}: extractiva/comparativa sin ancla_texto")
            elif len(ancla.split()) > 40:
                problemas.append(f"{pid}: ancla de {len(ancla.split())} palabras.")
        if not p.get("herramienta_esperada"):
            problemas.append(f"{pid}: sin herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"hacen falta 20 preguntas, hay {len(preguntas)}")
        n_comp = sum(p.get("familia") == "comparativa" for p in preguntas)
        if n_comp < 6:
            problemas.append(f"hacen falta 6 comparativas, hay {n_comp}")
    return problemas


problemas = validar(golden, exigir_20=True)
print("\nValidación:")
print("\n".join(f"  - {p}" for p in problemas) or "  sin problemas")


golden_set.jsonl: 20 preguntas
familia
numerica       7
extractiva     7
comparativa    6

Validación:
  sin problemas


## Tabla baseline contra final

Esta celda **gasta llamadas reales al modelo**: 20 preguntas por el agente baseline + 20
por el agente final, más una llamada de reescritura por cada búsqueda del sistema final.
Con `google/gemini-3.8-flash` son del orden de unos pocos céntimos en total, pero
ejecutadla solo cuando queráis los números definitivos para el informe — no en cada
prueba mientras depuráis.

Guarda `resultados_baseline.csv` y `resultados_final.csv` (el baseline etiquetado, como
pide el enunciado, **antes** de haber estado iterando sobre el final). Las respuestas
del agente van aparte, en `resultados_baseline_respuestas.csv` y
`resultados_final_respuestas.csv`.


In [57]:
if HAY_CLAVE:
    print("Evaluando BASELINE (día 10: sin guardrails, retrieval sin mejorar)...")
    t_base = evaluar("golden_set.jsonl", agente=agente_baseline, etiqueta="base",
                     salida_respuestas="resultados_baseline_respuestas.csv")
    t_base.to_csv("resultados_baseline.csv", index=False)

    print("Evaluando FINAL (guardrails + retrieval mejorado)...")
    t_fin = evaluar("golden_set.jsonl", agente=agente_final, etiqueta="fin",
                    salida_respuestas="resultados_final_respuestas.csv")
    t_fin.to_csv("resultados_final.csv", index=False)

    comparacion = pd.DataFrame([resumir(t_base, "baseline"), resumir(t_fin, "final")])
    print("\n" + comparacion.round(3).to_string(index=False))
    print("\nPor pregunta, la versión final:")
    print(t_fin.to_string(index=False))
else:
    print("Sin clave: el código está listo. Ejecutad esta celda con vuestra clave "
          "antes de entregar — es la tabla que va en el informe.")


Evaluando BASELINE (día 10: sin guardrails, retrieval sin mejorar)...
Evaluando FINAL (guardrails + retrieval mejorado)...
                                        
 versión  n  errores  reescrituras fallidas  cita  cifra  trayectoria  recall@5  coste medio (¢)  latencia media (s)  llamadas/pregunta
baseline 20        1                      0 0.714  0.538          1.0     0.154            1.728              28.459               3.60
   final 20        1                      0 0.929  0.923          1.0     0.692            1.331              26.832               2.65

Por pregunta, la versión final:
   id     familia ticker  intentos  latencia_s  coste_usd  llamadas  reescrituras_fallidas  cifra_obtenida fuente           chunk_id                                                                                           herramientas  cita cifra  trayectoria recall                                                                  error
g-n01    numerica   NVDA         1    6.130143   0.00329